# Data enrichment

For all feature-length films released between 1980 and 2025, the top 10 films per genre and per year are enriched with a synopsis, and at least three principals of each film with biographical information.

Pipeline:

1. **Spark**: select the films and their principals from the silver layer.
2. **Gathering (outside Spark, plain Python)**: IMDb id -> English Wikipedia article through Wikidata (property P345 = IMDb ID), then the article introduction through the Wikipedia API. Raw answers are stored as JSON lines in `data/enrichment/raw/` (resumable cache).
3. **Spark**: read the raw files with an explicit schema, clean the text, join, add the lineage and the notes for missing information, write the enriched tables.

Source: Wikipedia (human-written, CC BY-SA). Every enriched record keeps its source URL and revision id.

In [ ]:
import sys
sys.path.insert(0, "../jobs")  # notebooks/ -> jobs/

import json
import time
from datetime import datetime, timezone

import requests
from pyspark.sql import Window, functions as F, types as T

from common import DATA_DIR, PROJECT_ROOT, get_spark, read_silver

spark = get_spark("imdb_data_enrichment", driver_memory="2g", cores=2)
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.shuffle.partitions", "200")

ENRICH_DIR = DATA_DIR / "enrichment"
RAW_DIR = ENRICH_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PROJECT_ROOT / "results" / "enrichment"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Parameters

In [ ]:
START_YEAR, END_YEAR = 1980, 2025
TOP_N = 10                 # films per (year, genre), ranked by number of IMDb votes
N_PRINCIPALS = 3           # principals per film (lowest IMDb "ordering" first)
PRINCIPAL_CATEGORIES = ["actor", "actress", "director", "writer", "producer", "composer"]

# polite use of the Wikimedia APIs: identified client, requests in series, pause, retries
USER_AGENT = ("ECE-BigData-IMDb-student-project/1.0 "
              "(https://github.com/oscarico92/Big-Data-Project-IMDB; educational use)")
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
WIKIPEDIA_API = "https://en.wikipedia.org/w/api.php"
WIKIDATA_BATCH = 200       # IMDb ids per SPARQL query
WIKIPEDIA_BATCH = 20       # titles per request (API limit for intro extracts)
PAUSE_SECONDS = 0.5
MAX_RETRIES = 5

## 1. Selection with Spark

Popularity = number of IMDb votes (same measure as the exploration section). A film with several genres is selected once per genre where it ranks in the top 10, and is fetched only once.

In [ ]:
titles = read_silver(spark, "title.basics")
ratings = read_silver(spark, "title.ratings")

films = (
    titles.filter(F.col("is_feature_film") & ~F.col("is_unreleased")
                  & F.col("startYear").between(START_YEAR, END_YEAR))
    .join(ratings, "tconst")
    .select("tconst", "primaryTitle", "startYear", "genres", "averageRating", "numVotes")
)

by_genre = Window.partitionBy("startYear", "genre").orderBy(F.desc("numVotes"), "tconst")
top_films = (
    films.withColumn("genre", F.explode("genres"))
    .withColumn("genre_rank", F.row_number().over(by_genre))
    .filter(F.col("genre_rank") <= TOP_N)
    .cache()
)

selected_films = (
    top_films.groupBy("tconst", "primaryTitle", "startYear", "averageRating", "numVotes")
    .agg(F.sort_array(F.collect_set("genre")).alias("top10_genres"))
    .cache()
)

by_order = Window.partitionBy("tconst").orderBy("ordering")
selected_principals = (
    read_silver(spark, "title.principals")
    .filter(F.col("category").isin(PRINCIPAL_CATEGORIES))
    .join(selected_films.select("tconst"), "tconst", "left_semi")
    .dropDuplicates(["tconst", "nconst"])
    .withColumn("principal_rank", F.row_number().over(by_order))
    .filter(F.col("principal_rank") <= N_PRINCIPALS)
    .join(read_silver(spark, "name.basics").select("nconst", "primaryName", "birthYear", "deathYear"),
          "nconst", "left")
    .select("tconst", "nconst", "principal_rank", "category", "primaryName", "birthYear", "deathYear")
    .cache()
)

print("(year, genre) groups:", top_films.select("startYear", "genre").distinct().count())
print("selected film-genre rows:", top_films.count())
print("distinct films:", selected_films.count())
print("film-principal rows:", selected_principals.count(),
      "| distinct people:", selected_principals.select("nconst").distinct().count())

## 2. Gathering from Wikidata and Wikipedia (outside Spark)

Only the list of ids leaves Spark. Each answer is appended to a JSON lines file, and ids already in the file are skipped: the gathering can be stopped and restarted without calling the APIs again.

In [ ]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT, "Accept": "application/json"})


def call(url, params=None, data=None):
    """GET/POST with retries and exponential backoff on 429 / 5xx / network errors."""
    for attempt in range(MAX_RETRIES):
        try:
            response = (session.post(url, data=data, timeout=60) if data is not None
                        else session.get(url, params=params, timeout=60))
            if response.status_code == 200:
                time.sleep(PAUSE_SECONDS)
                return response.json()
            wait = int(response.headers.get("Retry-After", 2 ** (attempt + 1)))
            print(f"  HTTP {response.status_code}, retry in {wait}s")
        except requests.RequestException as error:
            wait = 2 ** (attempt + 1)
            print(f"  {type(error).__name__}, retry in {wait}s")
        time.sleep(wait)
    raise RuntimeError(f"giving up on {url}")


def already_done(path, key):
    if not path.exists():
        return set()
    with path.open(encoding="utf-8") as f:
        return {json.loads(line)[key] for line in f if line.strip()}


def append(path, records):
    with path.open("a", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

### 2.1 IMDb id -> English Wikipedia article (Wikidata P345)

In [ ]:
LINKS = RAW_DIR / "wikidata_links.jsonl"


def link_imdb_ids(imdb_ids):
    todo = sorted(set(imdb_ids) - already_done(LINKS, "imdb_id"))
    print(f"{len(todo)} ids to look up")
    for start in range(0, len(todo), WIKIDATA_BATCH):
        batch = todo[start:start + WIKIDATA_BATCH]
        values = " ".join(f'"{i}"' for i in batch)
        query = f"""
        SELECT ?imdb ?item ?article WHERE {{
          VALUES ?imdb {{ {values} }}
          ?item wdt:P345 ?imdb .
          OPTIONAL {{ ?article schema:about ?item ;
                               schema:isPartOf <https://en.wikipedia.org/> . }}
        }}"""
        answer = call(WIKIDATA_SPARQL, data={"query": query, "format": "json"})
        found = {}
        for row in answer["results"]["bindings"]:
            imdb = row["imdb"]["value"]
            if imdb not in found or "article" in row:
                found[imdb] = row
        records = []
        for imdb in batch:  # one record per id, also when nothing was found
            row = found.get(imdb, {})
            records.append({
                "imdb_id": imdb,
                "wikidata_item": row.get("item", {}).get("value"),
                "wikipedia_url": row.get("article", {}).get("value"),
                "retrieved_at": now(),
            })
        append(LINKS, records)
        print(f"  {start + len(batch)}/{len(todo)}")


film_ids = [r.tconst for r in selected_films.select("tconst").collect()]
person_ids = [r.nconst for r in selected_principals.select("nconst").distinct().collect()]
link_imdb_ids(film_ids + person_ids)

### 2.2 Article introduction (Wikipedia API, plain text)

In [ ]:
EXTRACTS = RAW_DIR / "wikipedia_extracts.jsonl"


def fetch_extracts(urls):
    titles = {requests.utils.unquote(u.rsplit("/wiki/", 1)[1]).replace("_", " "): u for u in urls}
    EXTRACTS.touch()  # the file exists even when there is nothing to fetch
    todo = sorted(set(titles) - already_done(EXTRACTS, "requested_title"))
    print(f"{len(todo)} articles to fetch")
    for start in range(0, len(todo), WIKIPEDIA_BATCH):
        batch = todo[start:start + WIKIPEDIA_BATCH]
        answer = call(WIKIPEDIA_API, params={
            "action": "query", "format": "json", "formatversion": 2, "redirects": 1,
            "prop": "extracts|info", "inprop": "url", "exintro": 1, "explaintext": 1,
            "exlimit": WIKIPEDIA_BATCH, "titles": "|".join(batch), "maxlag": 5,
        })
        query = answer.get("query", {})
        # requested title -> final title (normalization and redirects)
        final = {t: t for t in batch}
        for step in ("normalized", "redirects"):
            for m in query.get(step, []):
                for t, current in final.items():
                    if current == m["from"]:
                        final[t] = m["to"]
        pages = {p["title"]: p for p in query.get("pages", [])}
        records = []
        for t in batch:
            page = pages.get(final[t], {})
            records.append({
                "requested_title": t,
                "wikipedia_url": titles[t],
                "page_title": page.get("title"),
                "page_url": page.get("fullurl"),
                "revision_id": page.get("lastrevid"),
                "extract": page.get("extract"),
                "missing": bool(page.get("missing", not page)),
                "retrieved_at": now(),
            })
        append(EXTRACTS, records)
        print(f"  {start + len(batch)}/{len(todo)}")


with LINKS.open(encoding="utf-8") as f:
    article_urls = sorted({r["wikipedia_url"] for r in map(json.loads, f) if r["wikipedia_url"]})
fetch_extracts(article_urls)

## 3. Cleaning, schema enforcement and joins with Spark

The raw files are read with an explicit schema. The text is cleaned (whitespace, empty strings, pronunciation brackets left empty by the plain-text export). Every output row keeps its lineage (`source_url`, `wikipedia_revision_id`, `wikidata_item`, `retrieved_at`) and a `note` explaining why information is missing.

In [ ]:
LINK_SCHEMA = T.StructType([
    T.StructField("imdb_id", T.StringType(), False),
    T.StructField("wikidata_item", T.StringType()),
    T.StructField("wikipedia_url", T.StringType()),
    T.StructField("retrieved_at", T.StringType()),
])
EXTRACT_SCHEMA = T.StructType([
    T.StructField("requested_title", T.StringType(), False),
    T.StructField("wikipedia_url", T.StringType()),
    T.StructField("page_title", T.StringType()),
    T.StructField("page_url", T.StringType()),
    T.StructField("revision_id", T.LongType()),
    T.StructField("extract", T.StringType()),
    T.StructField("missing", T.BooleanType()),
    T.StructField("retrieved_at", T.StringType()),
])


def clean_text(col):
    text = F.regexp_replace(col, r"\(\s*[;,]?\s*\)", "")      # "( ; )" left by removed pronunciations
    text = F.regexp_replace(text, r"\s+", " ")
    text = F.trim(text)
    return F.when(F.length(text) > 0, text)


links = (spark.read.schema(LINK_SCHEMA).json(str(LINKS))
         .dropDuplicates(["imdb_id"]))
extracts = (spark.read.schema(EXTRACT_SCHEMA).json(str(EXTRACTS))
            .dropDuplicates(["wikipedia_url"])
            .select("wikipedia_url", "page_url", "revision_id", "missing",
                    clean_text(F.col("extract")).alias("text"),
                    F.col("retrieved_at").alias("text_retrieved_at")))

wiki = links.join(extracts, "wikipedia_url", "left")


def with_note(df, what):
    return df.withColumn("note", F.when(F.col("wikidata_item").isNull(),
                                        F.lit("no Wikidata item with this IMDb id"))
                         .when(F.col("wikipedia_url").isNull(),
                               F.lit("no English Wikipedia article for this entity"))
                         .when(F.col("missing") | F.col("text").isNull(),
                               F.lit(f"Wikipedia article has no introduction text ({what})")))


film_enriched = with_note(
    selected_films.join(wiki.withColumnRenamed("imdb_id", "tconst"), "tconst", "left"), "synopsis"
).select(
    "tconst", "primaryTitle", "startYear", "top10_genres", "averageRating", "numVotes",
    F.col("text").alias("synopsis"),
    F.coalesce("page_url", "wikipedia_url").alias("source_url"),
    F.col("revision_id").alias("wikipedia_revision_id"), "wikidata_item",
    F.coalesce("text_retrieved_at", "retrieved_at").alias("retrieved_at"),
    F.lit("Wikipedia (CC BY-SA 4.0), article introduction").alias("source"), "note",
)

principal_enriched = with_note(
    selected_principals.join(wiki.withColumnRenamed("imdb_id", "nconst"), "nconst", "left"), "biography"
).select(
    "tconst", "nconst", "principal_rank", "category", "primaryName", "birthYear", "deathYear",
    F.col("text").alias("biography"),
    F.coalesce("page_url", "wikipedia_url").alias("source_url"),
    F.col("revision_id").alias("wikipedia_revision_id"), "wikidata_item",
    F.coalesce("text_retrieved_at", "retrieved_at").alias("retrieved_at"),
    F.lit("Wikipedia (CC BY-SA 4.0), article introduction").alias("source"), "note",
)

film_enriched.write.mode("overwrite").parquet(str(ENRICH_DIR / "films"))
principal_enriched.write.mode("overwrite").parquet(str(ENRICH_DIR / "principals"))
top_films.select("startYear", "genre", "genre_rank", "tconst").write.mode("overwrite") \
    .parquet(str(ENRICH_DIR / "top10_by_year_genre"))

film_enriched = spark.read.parquet(str(ENRICH_DIR / "films"))
principal_enriched = spark.read.parquet(str(ENRICH_DIR / "principals"))
film_enriched.printSchema()

## Results

In [ ]:
coverage = spark.createDataFrame([
    ("films", film_enriched.count(),
     film_enriched.filter(F.col("synopsis").isNotNull()).count()),
    ("film principals", principal_enriched.count(),
     principal_enriched.filter(F.col("biography").isNotNull()).count()),
], "entity string, rows long, with_text long").withColumn(
    "coverage_pct", F.round(100 * F.col("with_text") / F.col("rows"), 2))
coverage.show()
coverage.toPandas().to_csv(RESULTS_DIR / "coverage.csv", index=False)

notes = (film_enriched.groupBy(F.lit("film").alias("entity"), "note").count()
         .unionByName(principal_enriched.groupBy(F.lit("principal").alias("entity"), "note").count())
         .orderBy("entity", F.desc("count")))
notes.show(truncate=False)
notes.toPandas().to_csv(RESULTS_DIR / "missing_notes.csv", index=False)

bios_per_film = (principal_enriched.groupBy("tconst")
                 .agg(F.count("biography").alias("principals_with_biography"))
                 .groupBy("principals_with_biography").count().orderBy("principals_with_biography"))
bios_per_film.show()
bios_per_film.toPandas().to_csv(RESULTS_DIR / "biographies_per_film.csv", index=False)

by_year = (film_enriched.groupBy("startYear")
           .agg(F.count("*").alias("films"), F.count("synopsis").alias("with_synopsis"))
           .withColumn("coverage_pct", F.round(100 * F.col("with_synopsis") / F.col("films"), 1))
           .orderBy("startYear"))
by_year.show(46)
by_year.toPandas().to_csv(RESULTS_DIR / "coverage_by_year.csv", index=False)

In [ ]:
# sample of the enriched records (the full tables stay in data/enrichment/, ignored by Git)
sample = (film_enriched.orderBy(F.desc("numVotes")).limit(20)
          .withColumn("synopsis", F.substring("synopsis", 1, 300)))
sample.select("tconst", "primaryTitle", "startYear", "synopsis", "source_url", "wikipedia_revision_id") \
    .show(5, truncate=80)
sample.toPandas().to_csv(RESULTS_DIR / "sample_films.csv", index=False)

(principal_enriched.join(sample.select("tconst"), "tconst", "left_semi")
 .withColumn("biography", F.substring("biography", 1, 300))
 .orderBy("tconst", "principal_rank")
 .toPandas().to_csv(RESULTS_DIR / "sample_principals.csv", index=False))